# F5-TTS Voice Cloning: In-Context vs Parametric Learning — Colab Edition

When does LoRA fine-tuning add value over zero-shot anchor conditioning in F5-TTS?

**Designed for Google Colab with H100 GPU runtime.**

**How to run:**
1. Set runtime to **H100 GPU**: `Runtime > Change runtime type > H100`
2. Run the setup cells to mount Drive and upload your audio
3. Set `AUDIO_FILE` in the config cell to your uploaded file
4. `Runtime > Run all`

**Structure:**
1. Setup & Configuration (Colab environment, data upload, dependencies)
2. Data Pipeline & Model Initialization
3. Training & Inference Functions
4. Grid Training — one LoRA adapter per training data amount
5. Grid Evaluation — inference across all (anchor duration x training data) combinations
6. Results — heatmap, line plots, audio samples

---
## 1. Setup & Configuration

### Steps
1. **Check GPU** and mount Google Drive (next cell)
2. **Install dependencies** (pre-installed torch is used)
3. **Upload your audio** — a `.m4a` or `.wav` recording of someone reading the 20 Harvard sentences
4. **Set `AUDIO_FILE`** below to point to your uploaded file
5. **Run all remaining cells** top-to-bottom

In [ ]:
# ── Verify GPU ──
import torch
if not torch.cuda.is_available():
    raise RuntimeError("No GPU detected! Go to Runtime > Change runtime type > H100 GPU")

gpu_name = torch.cuda.get_device_name(0)
gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {gpu_name} ({gpu_mem:.0f} GB VRAM)")

# ── Mount Google Drive (optional — for persistent storage of results) ──
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
print("Google Drive mounted at /content/drive")

In [ ]:
# torch and torchaudio are pre-installed on Colab
!pip install -q transformers peft vocos tqdm numpy librosa soundfile f5-tts stable-ts resemblyzer jiwer

In [ ]:
import os, shutil
os.makedirs('./data', exist_ok=True)

# ── Write Harvard sentences text file ──
harvard_text = """\
The birch canoe slid on the smooth planks.
Glue the sheet to the dark blue background.
It's easy to tell the depth of a well.
These days a chicken leg is a rare dish.
Rice is often served in round bowls.
The juice of lemons makes fine punch.
The box was thrown beside the parked truck.
The hogs were fed chopped corn and garbage.
Four hours of steady work faced us.
A large size in stockings is hard to sell.
The boy was there when the sun rose.
A rod is used to catch pink salmon.
The source of the huge river is the clear spring.
Kick the ball straight and follow through.
Help the woman get back to her feet.
A pot of tea helps to pass the evening.
Smoky fires lack flame and heat.
The soft cushion broke the man's fall.
The salt breeze came across from the sea.
The girl at the booth sold fifty bonds."""

with open('./data/harvard_sent_text.txt', 'w') as f:
    f.write(harvard_text)
print("Harvard sentences written to ./data/harvard_sent_text.txt")

# ── Upload your voice recording ──
# Record yourself (or someone) reading the 20 Harvard sentences above.
# Save as .m4a or .wav and upload here.
from google.colab import files
print("\nUpload your audio file (.m4a or .wav):")
uploaded = files.upload()
for filename in uploaded:
    dest = f'./data/{filename}'
    shutil.move(filename, dest)
    print(f"Saved: {dest}")
    print(f"\n--> Set AUDIO_FILE = \"./data/{filename}\" in the config cell below")

# Alternative: copy from Google Drive (uncomment and edit path)
# !cp "/content/drive/MyDrive/your_audio.m4a" ./data/

In [ ]:
from pathlib import Path

# ============================================================
# VOICE CONFIGURATION
# ============================================================

AUDIO_FILE = "./data/harvard_sentences_max.m4a"  # <-- Change to your uploaded filename
TEXT_FILE  = "./data/harvard_sent_text.txt"       # 20 Harvard sentences (auto-generated above)

# Auto-derive voice name from audio filename
_stem = Path(AUDIO_FILE).stem
VOICE_NAME = _stem.replace("harvard_sentences_", "")

# ============================================================
# GRID EXPERIMENT CONFIGURATION
# ============================================================
ANCHOR_DURATIONS = [0, 0.5, 1, 1.5, 2, 3, 4, 6, 8]   # seconds of anchor at inference
TRAIN_AMOUNTS = [0, 1, 5, 10, 20]                      # number of sentences for LoRA training

TEST_SENTENCES = [
    "The quick brown fox jumps over the lazy dog.",
    "I love anchors, they keep ships from drifting away.",
    "Please call Stella and ask her to bring these things.",
    "She had your dark suit in greasy wash water all year.",
    "The rainbow is a division of white light into many beautiful colors.",
]

# Training hyperparameters -- scale epochs inversely with data amount
EPOCHS_MAP = {1: 60, 5: 30, 10: 20, 20: 15}
LR = 7e-5
GRAD_ACCUM = 4
EVAL_EVERY = 3         # Evaluate speaker similarity every N epochs

# Generation settings
NFE_STEPS = 32
CFG_STRENGTH = 2.0
SWAY_COEF = -1.0

# Reproducibility -- multiple seeds for confidence intervals
SEEDS = [42, 137, 2024]

print(f"Voice: {VOICE_NAME} | Audio: {AUDIO_FILE}")
print(f"Anchor durations: {ANCHOR_DURATIONS}")
print(f"Training amounts: {TRAIN_AMOUNTS}")
print(f"Epochs per amount: {EPOCHS_MAP}")
print(f"Seeds: {SEEDS}")
print(f"Grid size: {len(ANCHOR_DURATIONS)} x {len(TRAIN_AMOUNTS)} = {len(ANCHOR_DURATIONS) * len(TRAIN_AMOUNTS)} cells")
print(f"Eval calls per cell: {len(TEST_SENTENCES)} sentences x {len(SEEDS)} seeds = {len(TEST_SENTENCES) * len(SEEDS)}")
print(f"Test sentences: {len(TEST_SENTENCES)}")

In [ ]:
import os, io, math, random, copy
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
import torchaudio
from peft import LoraConfig, get_peft_model
from vocos import Vocos
import IPython.display as ipd
from tqdm.auto import tqdm
import librosa
import soundfile as sf
import matplotlib.pyplot as plt

DEVICE = torch.device('cuda')
DTYPE = torch.float32  # Changed to float32 to prevent internal F5-TTS dtype mismatches

def show_fig(fig):
    """Display a matplotlib figure inline and close it."""
    plt.show()
    plt.close(fig)

print(f"Device: {DEVICE} | Dtype: {DTYPE}")

---
## 2. Data Pipeline & Model Initialization

In [ ]:
import stable_whisper

MAX_SEGMENT_SECONDS = 30
MAX_SENTENCES = 20

def align_audio(audio_path, sentences, sr=24000):
    """Whisper forced alignment -> returns (sentence_segments, sentence_texts, word_segments, word_texts, full_audio, word_timestamps)."""
    y, _ = librosa.load(audio_path, sr=sr, mono=True)
    y_16k, _ = librosa.load(audio_path, sr=16000, mono=True)

    full_text = " ".join(sentences)
    model = stable_whisper.load_model("base")
    result = model.align(y_16k, full_text, language="en")

    # Extract word-level timestamps
    words = []
    for segment in result.segments:
        for word in segment.words:
            words.append({"word": word.word.strip(), "start": word.start, "end": word.end})

    # ── Word-level segments ──
    pad = int(0.05 * sr)
    word_segments, word_texts = [], []
    for w in words:
        s = max(0, int(w["start"] * sr) - pad)
        e = min(len(y), int(w["end"] * sr) + pad)
        word_segments.append(torch.from_numpy(y[s:e].copy()))
        word_texts.append(w["word"])

    # ── Sentence-level segments ──
    boundaries = []
    word_idx = 0
    for sent in sentences:
        sent_words = sent.split()
        if word_idx >= len(words):
            break
        sent_start = words[word_idx]["start"]
        sent_end = words[word_idx]["end"]
        for _ in range(len(sent_words)):
            if word_idx < len(words):
                sent_end = words[word_idx]["end"]
                word_idx += 1
        boundaries.append((sent_start, sent_end))

    max_samples = int(MAX_SEGMENT_SECONDS * sr)
    sent_segments = []
    for start_t, end_t in boundaries:
        s = max(0, int(start_t * sr) - pad)
        e = min(len(y), int(end_t * sr) + pad)
        seg = torch.from_numpy(y[s:e].copy())
        if len(seg) > max_samples:
            seg = seg[:max_samples]
        sent_segments.append(seg)

    w_dur = [len(s) / sr for s in word_segments]
    s_dur = [len(s) / sr for s in sent_segments]
    print(f"Words:     {len(word_segments)} segments ({min(w_dur):.2f}s - {max(w_dur):.2f}s)")
    print(f"Sentences: {len(sent_segments)} segments ({min(s_dur):.1f}s - {max(s_dur):.1f}s)")

    del model
    return sent_segments, sentences[:len(sent_segments)], word_segments, word_texts, y, words


class AudioDataset(Dataset):
    """Generic audio-text dataset from pre-aligned segments."""
    def __init__(self, segments, texts):
        n = min(len(segments), len(texts))
        self.segments = segments[:n]
        self.texts = texts[:n]

    def __len__(self):
        return len(self.segments)

    def __getitem__(self, idx):
        return {"audio": self.segments[idx], "text": self.texts[idx]}


# Load and align
with open(TEXT_FILE, 'r', encoding='utf-8') as f:
    _raw_texts = [l.strip() for l in f if l.strip()][:MAX_SENTENCES]

sent_segs, sent_txts, word_segs, word_txts, _full_audio, _word_timestamps = align_audio(AUDIO_FILE, _raw_texts)

sentence_dataset = AudioDataset(sent_segs, sent_txts)
word_dataset     = AudioDataset(word_segs, word_txts)

print(f"\nSentence dataset: {len(sentence_dataset)} items")
print(f"Word dataset:     {len(word_dataset)} items")

In [ ]:
from f5_tts.api import F5TTS
from f5_tts.model.utils import convert_char_to_pinyin

print("Loading F5-TTS DiT backbone...")
f5tts = F5TTS(device=str(DEVICE))
ema_model = f5tts.ema_model
ema_model.to(DTYPE)

mel_spec_module = ema_model.mel_spec
vocab_char_map = ema_model.vocab_char_map

lora_config = LoraConfig(
    r=8, lora_alpha=16,
    target_modules=["to_q", "to_k", "to_v", "to_out.0", "ff.0.0", "ff.2"],
    lora_dropout=0.05, bias="none"
)
lora_model = get_peft_model(ema_model.transformer, lora_config)
lora_model.print_trainable_parameters()

vocoder = f5tts.vocoder
vocoder.to(DEVICE).to(torch.float32)  # Vocoder needs float32, stays on GPU (H100 has plenty of VRAM)
for param in vocoder.parameters():
    param.requires_grad = False
print(f"Vocoder frozen ({DEVICE}).")

In [ ]:
# ── Speaker similarity evaluation (resemblyzer) ──
from resemblyzer import VoiceEncoder

class SpeakerEval:
    """Speaker similarity using resemblyzer embeddings (cosine similarity, 0-1)."""
    def __init__(self):
        self.encoder = VoiceEncoder("cpu")

    def similarity(self, audio_a, audio_b, sr=24000):
        a_16k = librosa.resample(np.asarray(audio_a, dtype=np.float32), orig_sr=sr, target_sr=16000)
        b_16k = librosa.resample(np.asarray(audio_b, dtype=np.float32), orig_sr=sr, target_sr=16000)
        ea = self.encoder.embed_utterance(a_16k)
        eb = self.encoder.embed_utterance(b_16k)
        return float(np.dot(ea, eb) / (np.linalg.norm(ea) * np.linalg.norm(eb)))

speaker_eval = SpeakerEval()
print("Speaker eval ready (resemblyzer).")

In [ ]:
# -- WER evaluation (Whisper + jiwer) --
from transformers import pipeline
import jiwer

print("Loading Whisper for WER evaluation...")
whisper_pipe = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-large-v3-turbo",
    torch_dtype=torch.float16,
    device="cuda",
    chunk_length_s=30,  # <-- Added this to fix the KeyError: 'num_frames' issue
)

def compute_wer(audio_np, reference_text, sr=24000):
    """Transcribe audio with Whisper and compute WER against reference."""
    import tempfile, os
    with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as f:
        sf.write(f.name, audio_np, sr)
        result = whisper_pipe(f.name)
        hypothesis = result["text"].strip()
        os.unlink(f.name)
    wer_val = jiwer.wer(reference_text.lower(), hypothesis.lower())
    return wer_val, hypothesis

print("WER eval ready (Whisper large-v3-turbo).")

---
## 3. Training & Inference Functions

In [ ]:
def train_flow_matching(ema_model, dataset, epochs=15, lr=LR, save_dir="./adapters",
                        grad_accum_steps=GRAD_ACCUM, warmup_frac=0.1, max_loss_clip=1.5,
                        eval_every=EVAL_EVERY, verbose=True):
    """Fine-tune LoRA adapters using CFM flow matching loss.

    Every `eval_every` epochs, runs inference and computes speaker similarity
    (resemblyzer). Saves the checkpoint with the best speaker similarity score.
    """
    trainable_params = [p for p in ema_model.parameters() if p.requires_grad]
    n_samples = len(dataset)
    optimizer_steps_per_epoch = math.ceil(n_samples / grad_accum_steps)
    total_optimizer_steps = epochs * optimizer_steps_per_epoch
    warmup_steps = max(1, int(total_optimizer_steps * warmup_frac))

    if verbose:
        print(f"Training {sum(p.numel() for p in trainable_params):,} LoRA params")
        print(f"Samples: {n_samples} | Effective batch: {grad_accum_steps} | {optimizer_steps_per_epoch} opt steps/epoch")
        print(f"Schedule: {warmup_steps} warmup -> cosine decay over {total_optimizer_steps} steps")
        print(f"Speaker eval every {eval_every} epochs")

    optimizer = torch.optim.AdamW(trainable_params, lr=lr, weight_decay=0.01)

    def lr_lambda(step):
        if step < warmup_steps:
            return step / warmup_steps
        progress = (step - warmup_steps) / max(total_optimizer_steps - warmup_steps, 1)
        return 0.5 * (1 + math.cos(math.pi * progress))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    ema_model.train()
    global_step = 0
    best_sim = -1.0
    os.makedirs(save_dir, exist_ok=True)

    all_step_losses = []
    epoch_avg_losses = []
    spk_sim_history = []  # (epoch, similarity)

    for epoch in range(epochs):
        epoch_losses = []
        indices = list(range(n_samples))
        random.shuffle(indices)
        if verbose:
            print(f"\n--- Epoch {epoch+1}/{epochs} ---")
        optimizer.zero_grad()

        for i, idx in enumerate(tqdm(indices, desc=f"Epoch {epoch+1}", disable=not verbose)):
            sample = dataset[idx]
            waveform = sample['audio'].unsqueeze(0).to(DEVICE).to(DTYPE)
            text_list = convert_char_to_pinyin([sample['text']])

            loss, cond, pred = ema_model(inp=waveform, text=text_list)
            raw_loss = loss.item()
            if raw_loss > max_loss_clip:
                loss = loss * (max_loss_clip / raw_loss)
            (loss / grad_accum_steps).backward()
            epoch_losses.append(raw_loss)
            all_step_losses.append(raw_loss)

            if (i + 1) % grad_accum_steps == 0 or (i + 1) == n_samples:
                torch.nn.utils.clip_grad_norm_(trainable_params, max_norm=1.0)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad()

            global_step += 1

        avg_loss = np.mean(epoch_losses)
        epoch_avg_losses.append(avg_loss)
        current_lr = scheduler.get_last_lr()[0]

        # Periodic speaker similarity evaluation
        if (epoch + 1) % eval_every == 0 or epoch == epochs - 1:
            ema_model.eval()
            with torch.no_grad():
                eval_audio, _ = synthesize_speech(
                    ema_model, REF_TEXT, TEST_SENTENCES[0], ANCHOR_PATH,
                    nfe_steps=16, verbose=False
                )
                sim = speaker_eval.similarity(eval_audio, y_anchor)
            ema_model.train()
            spk_sim_history.append((epoch + 1, sim))

            if verbose:
                print(f"  FM loss: {avg_loss:.4f} | LR: {current_lr:.2e} | Speaker sim: {sim:.3f}")
            if sim > best_sim:
                best_sim = sim
                lora_model.save_pretrained(save_dir)
                if verbose:
                    print(f"  -> New best speaker similarity — saved checkpoint")
        else:
            if verbose:
                print(f"  FM loss: {avg_loss:.4f} | LR: {current_lr:.2e}")

    if verbose:
        print(f"\nTraining complete! Best speaker similarity: {best_sim:.3f}")

    return {
        "step_losses": all_step_losses,
        "epoch_avg_losses": epoch_avg_losses,
        "spk_sim_history": spk_sim_history,
        "best_sim": best_sim,
    }


def plot_loss(result, title="Training Loss"):
    """Plot per-step FM loss, per-epoch FM loss, and speaker similarity."""
    has_spk = len(result.get("spk_sim_history", [])) > 0
    ncols = 3 if has_spk else 2
    fig, axes = plt.subplots(1, ncols, figsize=(5 * ncols, 4))

    ax = axes[0]
    steps = result["step_losses"]
    ax.plot(steps, alpha=0.2, color="tab:blue", label="Raw")
    window = max(1, len(steps) // 20)
    if len(steps) > window:
        smoothed = np.convolve(steps, np.ones(window)/window, mode="valid")
        ax.plot(range(window-1, len(steps)), smoothed, color="tab:blue", label=f"Smoothed (w={window})")
    ax.set_xlabel("Step"); ax.set_ylabel("Loss")
    ax.set_title(f"{title} — FM Per Step"); ax.legend(); ax.grid(True, alpha=0.3)

    ax = axes[1]
    epochs = result["epoch_avg_losses"]
    ax.plot(range(1, len(epochs)+1), epochs, "o-", color="tab:orange")
    ax.set_xlabel("Epoch"); ax.set_ylabel("Avg Loss")
    ax.set_title(f"{title} — FM Per Epoch"); ax.grid(True, alpha=0.3)

    if has_spk:
        ax = axes[2]
        ep, sims = zip(*result["spk_sim_history"])
        ax.plot(ep, sims, "s-", color="tab:green", markersize=8)
        ax.set_xlabel("Epoch"); ax.set_ylabel("Speaker Similarity")
        ax.set_title(f"{title} — Speaker Sim (resemblyzer)")
        ax.set_ylim(0, 1); ax.grid(True, alpha=0.3)

    plt.tight_layout()
    show_fig(fig)

print("Training and plotting functions defined.")

In [ ]:
@torch.no_grad()
def synthesize_speech(ema_model, ref_text, gen_text, anchor_audio_path,
                      anchor_duration_s=None, nfe_steps=NFE_STEPS,
                      cfg_strength=CFG_STRENGTH, sway_coef=SWAY_COEF,
                      speed=0.95, seed=None, verbose=True):
    """Synthesize speech using the F5-TTS CFM ODE solver.

    Args:
        anchor_duration_s: If set, truncate anchor to this many seconds.
                           0 = no reference audio (pure LoRA). None = full anchor.
        seed: Random seed for reproducible generation.
    """
    ema_model.eval()
    waveform_np, _ = librosa.load(anchor_audio_path, sr=24000, mono=True)

    # Truncate anchor to desired duration
    no_ref = False
    if anchor_duration_s is not None:
        if anchor_duration_s <= 0:
            no_ref = True
            # Keep a short dummy for conditioning tensor shape
            waveform_np = waveform_np[:int(0.5 * 24000)]
        else:
            max_samples = int(anchor_duration_s * 24000)
            waveform_np = waveform_np[:max_samples]

    # Match ref_text to actual anchor duration
    if anchor_duration_s is not None and anchor_duration_s > 0:
        ref_text = get_ref_text_for_duration(anchor_duration_s)
    elif no_ref:
        ref_text = ""

    waveform = torch.from_numpy(waveform_np).unsqueeze(0)

    target_rms = 0.1
    rms = torch.sqrt(torch.mean(torch.square(waveform)))
    if rms < target_rms:
        waveform = waveform * target_rms / rms

    cond_audio = waveform.to(DEVICE).to(DTYPE)

    if ref_text and not ref_text.endswith(". ") and not ref_text.endswith("."):
        ref_text = ref_text + ". "
    elif ref_text.endswith("."):
        ref_text = ref_text + " "

    # Transition text to prevent word skipping at the ref/gen boundary
    if ref_text.strip():
        text_str = ref_text + "... " + gen_text
    else:
        text_str = gen_text
    text_list = convert_char_to_pinyin([text_str])

    hop_length = 256
    ref_audio_len = cond_audio.shape[-1] // hop_length
    gen_text_len = len(gen_text.encode("utf-8"))
    local_speed = 0.3 if gen_text_len < 10 else speed

    if no_ref or not ref_text.strip():
        # No reference: estimate duration from text length alone
        gen_frames = int(gen_text_len * 15.0 / local_speed)
        duration = ref_audio_len + gen_frames
    else:
        ref_text_len = len(ref_text.encode("utf-8"))
        duration = ref_audio_len + int(ref_audio_len / ref_text_len * gen_text_len / local_speed)

    # Duration floor: ensure at least ~8 frames per character for generated portion
    min_frames_per_char = 8
    gen_duration_frames = duration - ref_audio_len
    min_gen_frames = int(gen_text_len * min_frames_per_char)
    if gen_duration_frames < min_gen_frames:
        duration = ref_audio_len + min_gen_frames

    if verbose:
        anchor_label = f"{anchor_duration_s}s" if anchor_duration_s is not None else "full"
        print(f"Anchor: {anchor_label} | Ref: {ref_audio_len} frames ({ref_audio_len * hop_length / 24000:.1f}s) | "
              f"Target: {duration} frames ({duration * hop_length / 24000:.1f}s) | "
              f"NFE={nfe_steps}, CFG={cfg_strength}, sway={sway_coef}")

    generated, trajectory = ema_model.sample(
        cond=cond_audio, text=text_list, duration=duration,
        steps=nfe_steps, cfg_strength=cfg_strength, sway_sampling_coef=sway_coef,
        no_ref_audio=no_ref, seed=seed,
    )

    generated = generated.to(torch.float32)[:, ref_audio_len:, :]
    generated_mel = generated.permute(0, 2, 1)
    out_waveform = vocoder.decode(generated_mel)
    if rms < target_rms:
        out_waveform = out_waveform * rms / target_rms
    return out_waveform.squeeze().cpu().numpy(), 24000


def reset_lora():
    """Reset LoRA weights to untrained state."""
    for name, param in lora_model.named_parameters():
        if "lora_A" in name:
            nn.init.kaiming_uniform_(param, a=math.sqrt(5))
        elif "lora_B" in name:
            nn.init.zeros_(param)
    print("LoRA weights reset.")


# -- Build ref_text lookup for variable anchor durations --
anchor_word_end_times = []
for wt in _word_timestamps:
    anchor_word_end_times.append(wt["end"])

def get_ref_text_for_duration(duration_s, tolerance=0.15):
    """Return ref_text matching the first duration_s seconds of anchor audio.
    Uses a tolerance to include words ending near the boundary."""
    if duration_s <= 0:
        return ""
    words = [wt["word"] for wt in _word_timestamps if wt["end"] <= duration_s + tolerance]
    if not words:
        return _word_timestamps[0]["word"]
    return " ".join(words)


# -- Build anchor by concatenating sentence segments until ~8-10s --
ANCHOR_PATH = f"./data/anchor_{VOICE_NAME}.wav"
anchor_segments = []
anchor_texts = []
anchor_duration = 0.0
TARGET_ANCHOR_DURATION = 8.0  # seconds

for i in range(len(sentence_dataset)):
    sample = sentence_dataset[i]
    seg_duration = len(sample["audio"]) / 24000
    if anchor_duration + seg_duration > TARGET_ANCHOR_DURATION and anchor_segments:
        break
    anchor_segments.append(sample["audio"])
    anchor_texts.append(sample["text"])
    anchor_duration += seg_duration

silence = torch.zeros(int(0.15 * 24000))  # 150ms gap
parts = []
for seg in anchor_segments:
    parts.append(seg)
    parts.append(silence)
y_anchor = torch.cat(parts[:-1]).numpy()
sf.write(ANCHOR_PATH, y_anchor, 24000)
REF_TEXT = " ".join(anchor_texts)

N_ANCHOR_SENTS = len(anchor_segments)
print(f"Anchor: {N_ANCHOR_SENTS} segments, {len(y_anchor)/24000:.1f}s")
print(f"Reference text: {REF_TEXT}")
print(f"LoRA training pool: sentences {N_ANCHOR_SENTS}–{len(sentence_dataset)-1} ({len(sentence_dataset) - N_ANCHOR_SENTS} available)")

# Verify ref_text truncation works
for dur in ANCHOR_DURATIONS:
    rt = get_ref_text_for_duration(dur) if dur > 0 else "(no ref)"
    print(f'  {dur}s anchor -> ref_text: "{rt[:60]}{"..." if len(rt) > 60 else ""}"' )

print("\nReady.")

---
## 4. Grid Training

Train one LoRA adapter per training data amount. Each starts from a fresh LoRA reset and trains for `EPOCHS` epochs on a subset of the sentence dataset. The adapter with the best speaker similarity is saved.

In [ ]:
trained_adapters = {}  # {n_sentences: save_dir}
training_results = {}  # {n_sentences: result_dict}

for n_sent in TRAIN_AMOUNTS:
    if n_sent == 0:
        print(f"=== {n_sent} sentences: baseline (no training) ===\n")
        continue
    epochs = EPOCHS_MAP.get(n_sent, 15)
    suffix = 's' if n_sent != 1 else ''
    print(f"=== Training on {n_sent} sentence{suffix} ({epochs} epochs) ===")
    reset_lora()
    ema_model.to(DTYPE)
    lora_model.to(DTYPE)
    subset = Subset(sentence_dataset, range(N_ANCHOR_SENTS, min(N_ANCHOR_SENTS + n_sent, len(sentence_dataset))))
    save_dir = f"./adapters_{VOICE_NAME}_grid_{n_sent}sent"
    result = train_flow_matching(
        ema_model, subset, epochs=epochs, save_dir=save_dir
    )
    trained_adapters[n_sent] = save_dir
    training_results[n_sent] = result
    plot_loss(result, title=f"{n_sent} sentence{suffix} ({epochs} epochs)")
    print()

print(f"Training complete. Adapters saved for: {list(trained_adapters.keys())} sentences.")

---
## 5. Grid Evaluation

For each (anchor duration x training amount) cell, generate all test sentences and compute average speaker similarity against `y_anchor`. This isolates the interaction between in-context conditioning (anchor) and parametric learning (LoRA).

In [ ]:
from peft import PeftModel

n_anchors = len(ANCHOR_DURATIONS)
n_trains = len(TRAIN_AMOUNTS)
n_sents = len(TEST_SENTENCES)
n_seeds = len(SEEDS)

# 4D arrays: (anchor_dur, train_amt, sentence, seed)
all_sims = np.zeros((n_anchors, n_trains, n_sents, n_seeds))
all_wers = np.zeros((n_anchors, n_trains, n_sents, n_seeds))

total_cells = n_anchors * n_trains
cell_count = 0

for j, n_sent in enumerate(TRAIN_AMOUNTS):
    if n_sent == 0:
        reset_lora()
    else:
        lora_model.load_adapter(trained_adapters[n_sent], adapter_name="default")
        print(f"Loaded adapter: {trained_adapters[n_sent]}")

    for i, dur in enumerate(ANCHOR_DURATIONS):
        cell_count += 1
        for s, prompt in enumerate(TEST_SENTENCES):
            for k, seed in enumerate(SEEDS):
                audio, sr = synthesize_speech(
                    ema_model, REF_TEXT, prompt, ANCHOR_PATH,
                    anchor_duration_s=dur, seed=seed, verbose=False
                )
                sim = speaker_eval.similarity(audio, y_anchor, sr=24000)
                wer_val, hyp = compute_wer(audio, prompt, sr=24000)
                all_sims[i, j, s, k] = sim
                all_wers[i, j, s, k] = wer_val

        cell_sims = all_sims[i, j].flatten()
        cell_wers = all_wers[i, j].flatten()
        print(f"  [{cell_count}/{total_cells}] {dur}s anchor, {n_sent} sent -> "
              f"sim={cell_sims.mean():.3f} (+/-{cell_sims.std():.3f}) | "
              f"WER={cell_wers.mean():.3f} (+/-{cell_wers.std():.3f})")

# Aggregate grids (mean and std over sentences x seeds)
sim_grid = all_sims.mean(axis=(2, 3))
sim_std_grid = all_sims.reshape(n_anchors, n_trains, -1).std(axis=2)
wer_grid = all_wers.mean(axis=(2, 3))
wer_std_grid = all_wers.reshape(n_anchors, n_trains, -1).std(axis=2)

print("\nGrid evaluation complete.")
print("\nSpeaker Similarity grid:")
print(sim_grid)
print("\nWER grid:")
print(wer_grid)

---
## 6. Results

**Hypothesis:** Fine-tuning matters more as anchor duration decreases. The heatmap should show:
- Bottom row (8s anchor): flat — zero-shot conditioning suffices
- Top row (0s anchor): strong positive slope — LoRA must carry speaker identity

### Heatmaps

Side-by-side speaker similarity, delta, and word error rate. The similarity heatmap uses a data-adaptive color range. The delta heatmap shows where LoRA helps (+blue) vs. hurts (-red).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 7))

# -- Left: Speaker Similarity (data-adaptive range) --
ax = axes[0]
margin = (sim_grid.max() - sim_grid.min()) * 0.1
vmin_sim = max(0, sim_grid.min() - margin)
vmax_sim = min(1, sim_grid.max() + margin)
im1 = ax.imshow(sim_grid[::-1], cmap="RdYlGn", aspect="auto", vmin=vmin_sim, vmax=vmax_sim)
ax.set_xticks(range(len(TRAIN_AMOUNTS)))
ax.set_xticklabels([f"{n} sent" if n > 0 else "none" for n in TRAIN_AMOUNTS])
ax.set_yticks(range(len(ANCHOR_DURATIONS)))
ax.set_yticklabels([f"{d}s" for d in ANCHOR_DURATIONS[::-1]])
ax.set_xlabel("Fine-tuning data (sentences)")
ax.set_ylabel("Anchor duration (seconds)")
ax.set_title(f"Speaker Similarity -- {VOICE_NAME}")

for i in range(len(ANCHOR_DURATIONS)):
    for j in range(len(TRAIN_AMOUNTS)):
        ri = len(ANCHOR_DURATIONS) - 1 - i
        val = sim_grid[i, j]
        std = sim_std_grid[i, j]
        color = "white" if val < (vmin_sim + vmax_sim) / 2 else "black"
        ax.text(j, ri, f"{val:.2f}\n({std:.2f})", ha="center", va="center",
                color=color, fontsize=7, fontweight="bold")
plt.colorbar(im1, ax=ax, label="Resemblyzer cosine similarity", shrink=0.8)

# -- Middle: Delta from no-training baseline --
ax = axes[1]
delta_grid = sim_grid - sim_grid[:, 0:1]
abs_max = max(abs(delta_grid.min()), abs(delta_grid.max()), 0.01)
im2 = ax.imshow(delta_grid[::-1], cmap="RdBu", aspect="auto", vmin=-abs_max, vmax=abs_max)
ax.set_xticks(range(len(TRAIN_AMOUNTS)))
ax.set_xticklabels([f"{n} sent" if n > 0 else "none" for n in TRAIN_AMOUNTS])
ax.set_yticks(range(len(ANCHOR_DURATIONS)))
ax.set_yticklabels([f"{d}s" for d in ANCHOR_DURATIONS[::-1]])
ax.set_xlabel("Fine-tuning data (sentences)")
ax.set_ylabel("Anchor duration (seconds)")
ax.set_title(f"Delta vs No-Training Baseline -- {VOICE_NAME}")

for i in range(len(ANCHOR_DURATIONS)):
    for j in range(len(TRAIN_AMOUNTS)):
        ri = len(ANCHOR_DURATIONS) - 1 - i
        val = delta_grid[i, j]
        color = "black" if abs(val) < abs_max * 0.5 else "white"
        ax.text(j, ri, f"{val:+.3f}", ha="center", va="center",
                color=color, fontsize=7, fontweight="bold")
plt.colorbar(im2, ax=ax, label="Delta similarity", shrink=0.8)

# -- Right: WER heatmap --
ax = axes[2]
margin_w = (wer_grid.max() - wer_grid.min()) * 0.1
vmin_wer = max(0, wer_grid.min() - margin_w)
vmax_wer = min(1, wer_grid.max() + margin_w)
im3 = ax.imshow(wer_grid[::-1], cmap="RdYlGn_r", aspect="auto", vmin=vmin_wer, vmax=vmax_wer)
ax.set_xticks(range(len(TRAIN_AMOUNTS)))
ax.set_xticklabels([f"{n} sent" if n > 0 else "none" for n in TRAIN_AMOUNTS])
ax.set_yticks(range(len(ANCHOR_DURATIONS)))
ax.set_yticklabels([f"{d}s" for d in ANCHOR_DURATIONS[::-1]])
ax.set_xlabel("Fine-tuning data (sentences)")
ax.set_ylabel("Anchor duration (seconds)")
ax.set_title(f"Word Error Rate -- {VOICE_NAME}")

for i in range(len(ANCHOR_DURATIONS)):
    for j in range(len(TRAIN_AMOUNTS)):
        ri = len(ANCHOR_DURATIONS) - 1 - i
        val = wer_grid[i, j]
        std = wer_std_grid[i, j]
        color = "white" if val > (vmin_wer + vmax_wer) / 2 else "black"
        ax.text(j, ri, f"{val:.2f}\n({std:.2f})", ha="center", va="center",
                color=color, fontsize=7, fontweight="bold")
plt.colorbar(im3, ax=ax, label="Word Error Rate", shrink=0.8)

plt.tight_layout()
show_fig(fig)

### Line Plots

Speaker similarity vs training data, one line per anchor duration. If the hypothesis holds, short-anchor lines should have steeper slopes (fine-tuning helps more when in-context signal is weaker).

In [ ]:
colors = plt.cm.viridis(np.linspace(0, 1, len(ANCHOR_DURATIONS)))

fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# -- Left: absolute speaker similarity with error bars --
ax = axes[0]
for i, (dur, color) in enumerate(zip(ANCHOR_DURATIONS, colors)):
    means = sim_grid[i, :]
    stds = sim_std_grid[i, :]
    ax.errorbar(TRAIN_AMOUNTS, means, yerr=stds, fmt="o-", color=color,
                label=f"{dur}s anchor", markersize=6, linewidth=2, capsize=3)
ax.set_xlabel("Fine-tuning data (sentences)")
ax.set_ylabel("Speaker Similarity")
ax.set_title(f"Speaker Similarity vs Training Data -- {VOICE_NAME}")
ax.legend(title="Anchor duration", fontsize=7, title_fontsize=8)
ax.set_ylim(0.4, 1.0)
ax.grid(True, alpha=0.3)

# -- Middle: delta vs no-training baseline --
ax = axes[1]
baseline_col = sim_grid[:, 0]
for i, (dur, color) in enumerate(zip(ANCHOR_DURATIONS, colors)):
    deltas = sim_grid[i, :] - baseline_col[i]
    ax.plot(TRAIN_AMOUNTS, deltas, "s-", color=color,
            label=f"{dur}s anchor", markersize=6, linewidth=2)
ax.axhline(y=0, color="gray", linestyle="--", linewidth=1)
ax.set_xlabel("Fine-tuning data (sentences)")
ax.set_ylabel("Delta vs no-training baseline")
ax.set_title(f"Marginal Value of Fine-tuning -- {VOICE_NAME}")
ax.legend(title="Anchor duration", fontsize=7, title_fontsize=8)
ax.grid(True, alpha=0.3)

# -- Right: WER vs training data with error bars --
ax = axes[2]
for i, (dur, color) in enumerate(zip(ANCHOR_DURATIONS, colors)):
    means = wer_grid[i, :]
    stds = wer_std_grid[i, :]
    ax.errorbar(TRAIN_AMOUNTS, means, yerr=stds, fmt="o-", color=color,
                label=f"{dur}s anchor", markersize=6, linewidth=2, capsize=3)
ax.set_xlabel("Fine-tuning data (sentences)")
ax.set_ylabel("Word Error Rate")
ax.set_title(f"WER vs Training Data -- {VOICE_NAME}")
ax.legend(title="Anchor duration", fontsize=7, title_fontsize=8)
ax.grid(True, alpha=0.3)

plt.tight_layout()
show_fig(fig)

### Results Table & Audio Samples

Generate audio for the four corner cases to compare perceptually:
- (0s anchor, no training) — worst case: no speaker info at all
- (0s anchor, 20 sent) — LoRA only: speaker info purely from parametric learning
- (8s anchor, no training) — anchor only: speaker info purely from in-context conditioning
- (8s anchor, 20 sent) — both: in-context + parametric combined

In [ ]:
# -- Results table: Speaker Similarity --
print(f"Voice: {VOICE_NAME}")
print(f"\nSpeaker Similarity (mean +/- std):")
print(f"\n{chr(32)*10}", end="")
for n in TRAIN_AMOUNTS:
    label = f"{n} sent" if n > 0 else "none"
    print(f" | {label:>12}", end="")
print()
print("-" * (10 + len(TRAIN_AMOUNTS) * 15))
for i, dur in enumerate(ANCHOR_DURATIONS):
    print(f"{dur:>8}s ", end="")
    for j in range(len(TRAIN_AMOUNTS)):
        print(f" | {sim_grid[i,j]:.3f}({sim_std_grid[i,j]:.2f})", end="")
    print()

print(f"\nWord Error Rate (mean +/- std):")
print(f"\n{chr(32)*10}", end="")
for n in TRAIN_AMOUNTS:
    label = f"{n} sent" if n > 0 else "none"
    print(f" | {label:>12}", end="")
print()
print("-" * (10 + len(TRAIN_AMOUNTS) * 15))
for i, dur in enumerate(ANCHOR_DURATIONS):
    print(f"{dur:>8}s ", end="")
    for j in range(len(TRAIN_AMOUNTS)):
        print(f" | {wer_grid[i,j]:.3f}({wer_std_grid[i,j]:.2f})", end="")
    print()

# -- Corner-case audio samples --
DEMO_PROMPT = TEST_SENTENCES[0]
corners = [
    (0,   0,  "0s anchor, no training"),
    (0,   max(TRAIN_AMOUNTS), f"0s anchor, {max(TRAIN_AMOUNTS)} sent training"),
    (8,   0,  "8s anchor, no training"),
    (8,   max(TRAIN_AMOUNTS), f"8s anchor, {max(TRAIN_AMOUNTS)} sent training"),
]

print(f"\n\nReference anchor ({len(y_anchor)/24000:.1f}s):")
ipd.display(ipd.Audio(y_anchor, rate=24000))

for dur, n_sent, label in corners:
    if n_sent == 0:
        reset_lora()
    else:
        lora_model.load_adapter(trained_adapters[n_sent], adapter_name="default")

    audio, sr = synthesize_speech(ema_model, REF_TEXT, DEMO_PROMPT, ANCHOR_PATH,
                                  anchor_duration_s=dur, seed=42)
    sim = speaker_eval.similarity(audio, y_anchor, sr=24000)
    wer_val, hyp = compute_wer(audio, DEMO_PROMPT, sr=24000)

    out_path = f"./data/{VOICE_NAME}_anchor{dur}s_train{n_sent}sent.wav"
    sf.write(out_path, audio, sr)
    print(f"\n{label} (sim={sim:.3f}, WER={wer_val:.3f}):")
    print(f"  Expected: {DEMO_PROMPT}")
    print(f"  Got:      {hyp}")
    ipd.display(ipd.Audio(audio, rate=sr))

---
## 7. Anchor Duration Degradation Analysis

In [ ]:
# -- Investigate why 2s > 4s > 8s --
reset_lora()

DIAG_PROMPT = TEST_SENTENCES[0]
diag_durations = [0.5, 1, 1.5, 2, 3, 4, 6, 8]

print("Anchor Duration Diagnostics")
print("=" * 80)

diag_results = []
for dur in diag_durations:
    audio, sr = synthesize_speech(ema_model, REF_TEXT, DIAG_PROMPT, ANCHOR_PATH,
                                  anchor_duration_s=dur, seed=42)
    sim = speaker_eval.similarity(audio, y_anchor, sr=24000)
    wer_val, hyp = compute_wer(audio, DIAG_PROMPT, sr=24000)
    actual_duration = len(audio) / sr
    ref_text = get_ref_text_for_duration(dur) if dur > 0 else ""

    diag_results.append({
        "anchor_s": dur, "sim": sim, "wer": wer_val, "hyp": hyp,
        "audio_len_s": actual_duration, "ref_text": ref_text,
    })
    print(f"\n{dur}s anchor:")
    rt_display = ref_text[:80] + ("..." if len(ref_text) > 80 else "")
    print(f"  Ref text ({len(ref_text)} chars): {rt_display}")
    print(f"  Similarity: {sim:.3f} | WER: {wer_val:.3f}")
    print(f"  Generated duration: {actual_duration:.1f}s")
    print(f"  Transcription: {hyp}")

# -- Plot: similarity and WER vs anchor duration --
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

durations = [r["anchor_s"] for r in diag_results]
sims_d = [r["sim"] for r in diag_results]
wers_d = [r["wer"] for r in diag_results]
audio_lens = [r["audio_len_s"] for r in diag_results]
ref_lens = [len(r["ref_text"]) for r in diag_results]

ax1.plot(durations, sims_d, "o-", color="tab:green", label="Speaker Similarity", linewidth=2, markersize=8)
ax1.set_xlabel("Anchor Duration (s)")
ax1.set_ylabel("Speaker Similarity", color="tab:green")
ax1.set_title("Similarity & WER vs Anchor Duration (no LoRA)")
ax1.grid(True, alpha=0.3)
ax1b = ax1.twinx()
ax1b.plot(durations, wers_d, "s--", color="tab:red", label="WER", linewidth=2, markersize=8)
ax1b.set_ylabel("WER", color="tab:red")
ax1.legend(loc="upper left"); ax1b.legend(loc="upper right")

# Ref text length vs generated audio duration
ax2.bar(range(len(durations)), ref_lens, alpha=0.6, label="Ref text length (chars)", color="tab:blue")
ax2.set_xticks(range(len(durations)))
ax2.set_xticklabels([f"{d}s" for d in durations])
ax2.set_xlabel("Anchor Duration")
ax2.set_ylabel("Ref text length (chars)", color="tab:blue")
ax2.set_title("Reference Text Length vs Anchor Duration")
ax2b = ax2.twinx()
ax2b.plot(range(len(durations)), audio_lens, "D-", color="tab:orange", label="Generated audio (s)", linewidth=2)
ax2b.set_ylabel("Generated audio duration (s)", color="tab:orange")
ax2.legend(loc="upper left"); ax2b.legend(loc="upper right")

plt.tight_layout()
show_fig(fig)

# -- Content leakage check --
print("\n\nContent Leakage Check:")
print("=" * 80)
print("Do transcriptions contain words from the reference text?")
ref_words = set(REF_TEXT.lower().split())
for r in diag_results:
    hyp_words = set(r["hyp"].lower().split())
    leaked = hyp_words & ref_words - set(DIAG_PROMPT.lower().split())
    leak_pct = len(leaked) / max(len(hyp_words), 1) * 100
    if leaked:
        dur_val = r["anchor_s"]
        print(f"  {dur_val}s: LEAKED {len(leaked)} words ({leak_pct:.0f}%): {leaked}")
    else:
        dur_val = r["anchor_s"]
        print(f"  {dur_val}s: No content leakage detected")

---
## 8. LoRA Rank Sweep

Does increasing LoRA capacity help encode speaker identity without an anchor? We test ranks [2, 4, 8, 16, 32] at the 0s anchor condition with 20 sentences of training data.

In [ ]:
from peft import LoraConfig, get_peft_model
import gc

RANKS = [2, 4, 8, 16, 32]
rank_results = []

for rank in RANKS:
    print("\n" + "=" * 60)
    print(f"LoRA rank = {rank}")
    print("=" * 60)

    # Remove old LoRA and apply new config
    ema_model.transformer = lora_model.unload()
    gc.collect()
    torch.cuda.empty_cache()

    new_lora_config = LoraConfig(
        r=rank, lora_alpha=rank * 2,
        target_modules=["to_q", "to_k", "to_v", "to_out.0", "ff.0.0", "ff.2"],
        lora_dropout=0.05, bias="none"
    )
    lora_model_new = get_peft_model(ema_model.transformer, new_lora_config)
    globals()["lora_model"] = lora_model_new
    lora_model_new.print_trainable_parameters()

    # Train on 20 sentences
    ema_model.to(DTYPE)
    lora_model_new.to(DTYPE)
    subset = Subset(sentence_dataset, range(N_ANCHOR_SENTS, min(N_ANCHOR_SENTS + 20, len(sentence_dataset))))
    save_dir = f"./adapters_{VOICE_NAME}_rank{rank}"
    epochs = EPOCHS_MAP.get(20, 15)

    result = train_flow_matching(
        ema_model, subset, epochs=epochs, save_dir=save_dir, verbose=False
    )
    n_params = sum(p.numel() for p in lora_model_new.parameters() if p.requires_grad)

    # Evaluate at 0s anchor
    sims_r, wers_r = [], []
    for prompt in TEST_SENTENCES:
        for seed in SEEDS:
            audio, sr = synthesize_speech(
                ema_model, REF_TEXT, prompt, ANCHOR_PATH,
                anchor_duration_s=0, seed=seed, verbose=False
            )
            sim = speaker_eval.similarity(audio, y_anchor, sr=24000)
            wer_val, _ = compute_wer(audio, prompt, sr=24000)
            sims_r.append(sim)
            wers_r.append(wer_val)

    avg_sim = np.mean(sims_r)
    avg_wer = np.mean(wers_r)
    rank_results.append((rank, n_params, avg_sim, np.std(sims_r), avg_wer, np.std(wers_r)))
    print(f"  r={rank}: {n_params:,} params | sim={avg_sim:.3f} (+/-{np.std(sims_r):.3f}) | WER={avg_wer:.3f}")

# -- Restore original r=8 LoRA config --
ema_model.transformer = lora_model_new.unload()
gc.collect(); torch.cuda.empty_cache()

lora_config = LoraConfig(
    r=8, lora_alpha=16,
    target_modules=["to_q", "to_k", "to_v", "to_out.0", "ff.0.0", "ff.2"],
    lora_dropout=0.05, bias="none"
)
lora_model = get_peft_model(ema_model.transformer, lora_config)

# -- Plot rank sweep --
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ranks_list = [r[0] for r in rank_results]
params_list = [r[1] for r in rank_results]
sims_list = [r[2] for r in rank_results]
sim_stds_list = [r[3] for r in rank_results]
wers_list = [r[4] for r in rank_results]
wer_stds_list = [r[5] for r in rank_results]

ax1.errorbar(ranks_list, sims_list, yerr=sim_stds_list, fmt="o-", color="tab:green",
             markersize=8, linewidth=2, capsize=4)
ax1.set_xlabel("LoRA Rank")
ax1.set_ylabel("Speaker Similarity (0s anchor)")
ax1.set_title("Does Higher LoRA Rank Encode Speaker Identity?")
ax1.set_ylim(0.4, 1.0)
ax1.grid(True, alpha=0.3)
ax1_top = ax1.twiny()
ax1_top.set_xlim(ax1.get_xlim())
ax1_top.set_xticks(ranks_list)
ax1_top.set_xticklabels([f"{p/1e6:.1f}M" for p in params_list], fontsize=8)
ax1_top.set_xlabel("Trainable Parameters", fontsize=9)

ax2.errorbar(ranks_list, wers_list, yerr=wer_stds_list, fmt="s-", color="tab:red",
             markersize=8, linewidth=2, capsize=4)
ax2.set_xlabel("LoRA Rank")
ax2.set_ylabel("WER (0s anchor)")
ax2.set_title("Intelligibility vs LoRA Rank (No Anchor)")
ax2.grid(True, alpha=0.3)

plt.tight_layout()
show_fig(fig)

print("\nLoRA Rank Sweep Summary (0s anchor, 20 sentences):")
print(f"  Rank |     Params |   Similarity |        WER")
print("-" * 50)
for r, p, s, ss, w, ws in rank_results:
    print(f"{r:>6} | {p:>10,} | {s:.3f} +/-{ss:.3f} | {w:.3f} +/-{ws:.3f}")

---
## 9. Findings

### 9.1 Anchor Duration Dominates Speaker Identity Transfer

The single most important factor for voice cloning quality is **anchor duration**, not fine-tuning.
With zero LoRA training, speaker similarity rises steeply from **0.59** (0s anchor) to **0.93** (1.5s anchor) —
a **+0.34** improvement from in-context conditioning alone. By contrast, the maximum gain from LoRA fine-tuning
(across all anchor durations and training amounts) is only **+0.025** (0.5s anchor, 5 sentences).
This means anchor conditioning provides roughly **14x more speaker identity signal** than parametric adaptation.

| Condition | Speaker Similarity | Change from baseline |
|---|---|---|
| 0s anchor, no training | 0.590 | — |
| 0s anchor, 20 sentences | 0.595 | +0.005 |
| 1.5s anchor, no training | 0.930 | +0.340 |
| 8s anchor, no training | 0.924 | +0.334 |
| 8s anchor, 20 sentences | 0.923 | +0.333 |

### 9.2 LoRA Fine-Tuning Provides Marginal and Inconsistent Gains

Across the 9x5 grid of (anchor duration x training data) conditions, LoRA fine-tuning yields negligible
improvements to speaker similarity. The maximum delta is **+0.025** at the 0.5s anchor with 5 sentences.
At the 0s anchor condition — where we hypothesized LoRA would matter most — training on 20 sentences
yields only **+0.005** similarity over baseline (0.595 vs. 0.590), well within the standard deviation of
0.05. At longer anchors (6-8s), fine-tuning sometimes *hurts* slightly (delta of -0.001 to -0.002),
confirming that when the in-context signal is strong, LoRA can introduce noise rather than improve quality.

### 9.3 Non-Monotonic Similarity Curve Reveals an Optimal Anchor Length

Speaker similarity does **not** increase monotonically with anchor duration. The diagnostics show a peak
at **1.5s** (sim = 0.930–0.935), followed by a dip at **2–3s** (sim = 0.898–0.889), before recovering
at **4–8s** (sim = 0.906–0.924). This suggests that F5-TTS's DiT architecture has a **sweet spot** around
1.5s where the reference audio is long enough to capture speaker characteristics but short enough
to avoid the duration-prediction challenges that arise with longer conditioning segments.
The dip at 2–3s correlates with word-boundary alignment issues in the reference text:
at 2s, the reference text is "The birch canoe" (mid-sentence truncation),
whereas at 1.5s the model receives only "The" and relies more on acoustic features than text alignment.

### 9.4 WER Reveals Catastrophic Repetition at 1.5s Anchor

While 1.5s achieves the **best** speaker similarity, it also produces the **worst** word error rate
(WER = 2.6–6.1, i.e., the generated text is repeated multiple times). The diagnostic transcription
at 1.5s shows *"the quick brown fox jumps over the lazy dog the quick brown fox jumps over the lazy dogs"* —
the model generates the sentence twice. This stems from a mismatch between the 1.5s anchor audio and
a very short reference text ("The"), causing the model to allocate far too many frames to the generation
target (1,436 frames = 15.3s for a sentence that should be ~4s). The over-allocated silence is filled
with repetition.

| Anchor | Similarity | WER | Failure Mode |
|---|---|---|---|
| 0.5s | 0.793 | 0.081 | Clean generation |
| 1.5s | 0.930 | 2.607 | Repetition (2x sentence) |
| 3s | 0.889 | 0.049 | Clean generation |
| 8s | 0.924 | 0.063 | Clean generation |

The **3s anchor** emerges as the practical sweet spot — it achieves the lowest WER (0.049) and
competitive similarity (0.889) without the repetition artifacts.

### 9.5 LoRA Rank Sweep: Capacity Alone Cannot Substitute for In-Context Conditioning

Increasing LoRA rank from 2 to 32 (630K to 10M trainable parameters — a 16x increase) at the 0s
anchor condition produces only a modest similarity improvement from **0.594 to 0.638** (+0.044).
Even rank-32 with 10M parameters (2.9% of the model) cannot approach the similarity levels achieved
by simply providing 1 second of anchor audio (0.904). This demonstrates a fundamental limitation:
**LoRA fine-tuning on 20 sentences cannot capture the acoustic features that in-context conditioning
extracts from even a brief audio sample.** Moreover, higher ranks slightly increase WER (0.203 at
rank 32 vs. 0.140 at rank 4), suggesting that larger adapters overfit to training data prosody rather
than learning generalizable speaker representations.

| Rank | Params | Similarity | WER |
|---|---|---|---|
| 2 | 630K | 0.594 | 0.281 |
| 4 | 1.26M | 0.598 | 0.140 |
| 8 | 2.52M | 0.600 | 0.154 |
| 16 | 5.05M | 0.618 | 0.179 |
| 32 | 10.1M | 0.638 | 0.203 |

### 9.6 Practical Implications

1. **For deployment:** Zero-shot F5-TTS with a 3–8s anchor achieves similarity > 0.89 with WER < 0.07,
   making LoRA fine-tuning unnecessary for most use cases. The marginal gains from fine-tuning do not
   justify the computational cost (60–75 optimization steps on H100) or the risk of WER degradation.

2. **Anchor length matters more than anchor quality tuning:** The difference between 3s and 8s anchors
   is only +0.035 similarity, but the 3s anchor produces the best WER. Users should prioritize
   providing **clean, representative** reference audio over maximizing duration.

3. **LoRA's value proposition is narrow:** Fine-tuning may help in the 0.5–1s anchor regime where
   deltas of +0.01–0.025 are observed, but this is precisely the regime where users should instead
   provide slightly more reference audio for a much larger benefit.

4. **The repetition failure mode at 1.5s** highlights the importance of reference text alignment —
   when the ratio of anchor frames to reference text tokens is too high, F5-TTS over-generates.
   Production systems should enforce anchor-duration / text-length constraints.

In [ ]:
import IPython.display as ipd

# 1. Load the 20-sentence fine-tuned adapter
print("Loading adapter for 20 sentences...")
lora_model.load_adapter(trained_adapters[20], adapter_name="default")

# 2. Define your custom sentence
custom_prompt = "I can say wathever I want with this voice."

# 3. Synthesize speech using the 8-second anchor
print("\nGenerating audio (8s anchor, 20-sentence model)...")
custom_audio, custom_sr = synthesize_speech(
    ema_model,
    REF_TEXT,
    custom_prompt,
    ANCHOR_PATH,
    anchor_duration_s=8
)

# 4. Display the result
print(f"\nPrompt: {custom_prompt}")
ipd.display(ipd.Audio(custom_audio, rate=custom_sr))